# PoC 2: RAG + Surprise Scoring — "What are you NOT seeing?"

Fetch R&S-relevant papers from arxiv, build a RAG index, extract technology drivers, and score them by "surprise" (embedding distance from consensus = potential blind spot).

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

from dotenv import load_dotenv
load_dotenv("../.env")

from openai import AzureOpenAI
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

client = AzureOpenAI(
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    api_version=os.environ["AZURE_OPENAI_API_VERSION"],
)
MODEL = os.environ.get("AZURE_OPENAI_DEPLOYMENT", "gpt-4o")
print(f"Connected: {MODEL}")

## 1. Fetch Papers from arxiv

In [ ]:
import arxiv

QUERIES = [
    "spectrum monitoring machine learning",
    "6G wireless communications future",
    "test and measurement automation RF",
    "signal processing cognitive radio",
    "electronic warfare spectrum",
    "LEO satellite RF sensing",
    "Open RAN disaggregated network",
]
MAX_PER_QUERY = 8

papers = []
seen_ids = set()

for query in QUERIES:
    search = arxiv.Search(query=query, max_results=MAX_PER_QUERY, sort_by=arxiv.SortCriterion.Relevance)
    for result in search.results():
        if result.entry_id not in seen_ids:
            seen_ids.add(result.entry_id)
            papers.append({
                "id": result.entry_id,
                "title": result.title,
                "abstract": result.summary,
                "published": result.published.strftime("%Y-%m-%d"),
                "query": query,
            })

print(f"Fetched {len(papers)} unique papers from {len(QUERIES)} queries")
pd.DataFrame(papers)[["title", "published", "query"]].head(10)

## 2. Build RAG Index (LlamaIndex + ChromaDB)

In [ ]:
import chromadb
from llama_index.core import VectorStoreIndex, Document, Settings
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

Settings.embed_model = HuggingFaceEmbedding(model_name="all-MiniLM-L6-v2")
Settings.llm = None  # we'll call GPT-4o directly, not through LlamaIndex

documents = [
    Document(
        text=f"Title: {p['title']}\n\nAbstract: {p['abstract']}",
        metadata={"title": p["title"], "published": p["published"], "arxiv_id": p["id"]},
    )
    for p in papers
]

chroma_client = chromadb.Client()
chroma_collection = chroma_client.get_or_create_collection("arxiv_papers")
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
index = VectorStoreIndex.from_documents(documents, vector_store=vector_store)
query_engine = index.as_query_engine(similarity_top_k=5)

print(f"Indexed {len(documents)} documents")

# Sanity check
test_result = query_engine.query("What are the latest trends in spectrum monitoring?")
print(f"\nSanity check query result (truncated):\n{str(test_result)[:300]}...")

## 3. Extract Technology Drivers via RAG + LLM

In [ ]:
from src.models import DriverExtractionResult

FORESIGHT_QUERIES = [
    "What emerging technologies will transform spectrum monitoring by 2035?",
    "What are the biggest disruptions coming to wireless test and measurement?",
    "How will AI change RF signal processing and cognitive radio?",
    "What new sensing modalities could replace traditional spectrum analyzers?",
    "What geopolitical or regulatory shifts could reshape the wireless industry?",
]

all_context_chunks = []
for q in FORESIGHT_QUERIES:
    retriever = index.as_retriever(similarity_top_k=4)
    nodes = retriever.retrieve(q)
    for node in nodes:
        all_context_chunks.append(node.get_text())

unique_chunks = list(set(all_context_chunks))
print(f"Retrieved {len(unique_chunks)} unique context chunks from {len(FORESIGHT_QUERIES)} queries")

context_block = "\n\n---\n\n".join(unique_chunks[:20])

response = client.beta.chat.completions.parse(
    model=MODEL,
    messages=[
        {"role": "system", "content": (
            "You are a technology foresight analyst specializing in wireless communications, "
            "spectrum management, and test & measurement (T&M) for a company like Rohde & Schwarz. "
            "Extract technology drivers from the provided research context. "
            "A driver is a technology trend, capability, or external force that could significantly "
            "shape the industry in the next 5-10 years. "
            "Include BOTH obvious/mainstream drivers AND surprising/non-obvious ones. "
            "Aim for 15-25 drivers covering the full STEEP spectrum."
        )},
        {"role": "user", "content": f"Extract technology drivers from this research context:\n\n{context_block}"},
    ],
    response_format=DriverExtractionResult,
    temperature=0.4,
)

extracted = response.choices[0].message.parsed
print(f"Extracted {len(extracted.drivers)} technology drivers")
pd.DataFrame([d.model_dump() for d in extracted.drivers])[["name", "steep_category", "trl_low", "trl_high"]]

## 4. Surprise Scoring

Embed each driver, compute centroid ("industry consensus"), rank by cosine distance. High distance = potential blind spot.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

embed_model = Settings.embed_model

driver_texts = [f"{d.name}: {d.description}" for d in extracted.drivers]
embeddings = np.array([embed_model.get_text_embedding(t) for t in driver_texts])

centroid = embeddings.mean(axis=0, keepdims=True)

similarities = cosine_similarity(embeddings, centroid).flatten()
surprise_scores = 1 - similarities

driver_df = pd.DataFrame([d.model_dump() for d in extracted.drivers])
driver_df["surprise_score"] = surprise_scores
driver_df = driver_df.sort_values("surprise_score", ascending=False)

print("Top 10 most 'surprising' (potential blind spots):")
driver_df[["name", "steep_category", "surprise_score"]].head(10)

## 5. Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# Bar chart: surprise scores
plot_df = driver_df.sort_values("surprise_score", ascending=True)
colors = plt.cm.RdYlGn_r(plot_df["surprise_score"] / plot_df["surprise_score"].max())
axes[0].barh(plot_df["name"], plot_df["surprise_score"], color=colors)
axes[0].set_xlabel("Surprise Score")
axes[0].set_title("Driver Surprise Scores\n(high = potential blind spot)")
axes[0].axvline(x=plot_df["surprise_score"].median(), color="gray", linestyle="--", alpha=0.5, label="median")
axes[0].legend()

# PCA scatter
from sklearn.decomposition import PCA
pca = PCA(n_components=2)
coords = pca.fit_transform(embeddings)
centroid_2d = pca.transform(centroid)

scatter = axes[1].scatter(coords[:, 0], coords[:, 1], c=surprise_scores, cmap="RdYlGn_r", s=80, edgecolors="black", linewidths=0.5)
axes[1].scatter(centroid_2d[0, 0], centroid_2d[0, 1], c="blue", marker="X", s=200, label="centroid (consensus)", zorder=5)

for i, name in enumerate(driver_df["name"]):
    idx = plot_df.index.get_loc(driver_df.index[i]) if driver_df.index[i] in plot_df.index else i
    axes[1].annotate(name, (coords[i, 0], coords[i, 1]), fontsize=7, alpha=0.8)

axes[1].set_title("Driver Embedding Space (PCA)\nFar from center = surprising")
axes[1].legend()
plt.colorbar(scatter, ax=axes[1], label="Surprise Score")
plt.tight_layout()
plt.show()

## Full Driver Table

In [ ]:
driver_df[["name", "description", "steep_category", "trl_low", "trl_high", "surprise_score"]].round(4)